In [1]:
# N-gramms
from core.config import CURRENT
from core.clean import clean_text

import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from nltk.corpus import stopwords

df = pd.read_csv(CURRENT["path"])
df["Text"] = df[CURRENT["text_col"]].apply(clean_text)
df = df.dropna(subset=[CURRENT["label_col"]])

train, test = train_test_split(df, test_size=0.2, random_state=42)
vect = TfidfVectorizer(
    stop_words=stopwords.words('german'),
    max_features=CURRENT["max_features"],
    ngram_range=(1, 2),
    sublinear_tf=True,
)

X_train = vect.fit_transform(train["Text"])
X_test = vect.transform(test["Text"])

y_train = train[CURRENT["label_col"]]
y_test = test[CURRENT["label_col"]]

model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

start = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start

y_pred = model.predict(X_test)

# Class-level metrics in a DataFrame, without "classification_report"
labels = sorted(set(y_test))
prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred, labels=labels, zero_division=0
)

df_report = pd.DataFrame({
    "Precision": prec,
    "Recall": rec,
    "F1": f1,
    "Support": sup,
}, index=[int(l) for l in labels])

print(f"Train time: {train_time:.2f}s")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(df_report.round(4))

Train time: 42.27s
Accuracy: 0.7103
F1 macro: 0.3829

   Precision  Recall      F1  Support
1     0.9697  0.7862  0.8684    69908
2     0.1265  0.4104  0.1934     4432
3     0.1635  0.3276  0.2182     1731
4     0.1950  0.2625  0.2238     2526
5     0.3278  0.3239  0.3258     3939
6     0.4273  0.5170  0.4679     3412
